# HAR regulatory genomics: Colab feasibility pilot

**Goal:** objectively check file availability, biological replicate metadata and HAR region overlap before designing a publication-level study.

**This notebook uses real source files only. It does not simulate biological findings.**

Data: [GSE270828](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE270828) · [GSE292632](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE292632) · [GSE305245](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE305245) · [Pal et al. Dryad](https://doi.org/10.5061/dryad.c59zw3rkj).

**Important:** 180 targeted HARs ≠ 180 independently validated HAR–gene pairs; 106 open-chromatin perturbations ≠ 106 HARs; postnatal tissue ≠ neural-stem-cell experimental validation.

Run all cells top to bottom in Google Colab. Files are downloaded to `/content/har_pilot` only when a source is accessible. If a source blocks automated downloads, you can upload a legitimate copy from the linked repository and rerun the analysis cells.


In [ ]:
# 1. Dependencies and reproducible configuration (small-file pilot only)
import os, re, io, json, gzip, hashlib, traceback, shutil, zipfile
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import requests
import numpy as np
import pandas as pd

BASE = Path('/content/har_pilot') if Path('/content').exists() else Path.cwd()/'har_pilot'
DATA = BASE/'data'; OUT = BASE/'outputs'
DATA.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True,exist_ok=True)
USER_AGENT = 'HAR-feasibility-audit/0.1 (public academic GEO and Dryad files)'
# DO NOT set this until the originating BED file's coordinate assembly has been confirmed.
VERIFIED_GSE292632_BED_BUILD = None # set to 'GRCh38' ONLY if confirmed from accompanying metadata
VERIFIED_PAL_HNSC_BUILD = 'GRCh38' # Pal Dryad human neural-stem-cell interactions
MAX_DOWNLOAD_MB = 12
SOURCES = {
 'GSE270828_guide_feature_readme':{
    'url':'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE270nnn/GSE270828/suppl/GSE270828_feature_README.csv',
    'filename':'GSE270828_feature_README.csv'},
 'GSE292632_target_regions':{
    'url':'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE292nnn/GSE292632/suppl/GSE292632_NPC_4N_CRISPRi_screen_target_regions.bed.gz',
    'filename':'GSE292632_NPC_4N_CRISPRi_screen_target_regions.bed.gz'},
 'Pal_hNSC_CHiC':{
    'url':'https://datadryad.org/downloads/file_stream/3922898',
    'filename':'HAR_interactions_hNSCs.txt'},
 'GSE270828_GEO_metadata':{
    'url':'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE270nnn/GSE270828/soft/GSE270828_family.soft.gz',
    'filename':'GSE270828_family.soft.gz'},
 'GSE292632_GEO_metadata':{
    'url':'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE292nnn/GSE292632/soft/GSE292632_family.soft.gz',
    'filename':'GSE292632_family.soft.gz'},
 'GSE305245_GEO_metadata':{
    'url':'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE305nnn/GSE305245/soft/GSE305245_family.soft.gz',
    'filename':'GSE305245_family.soft.gz'},
}
print('Pilot directory:', BASE)


In [ ]:
# 2. Download SMALL primary files, tracking every download or failure.
def sha256file(path, chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda: f.read(chunk), b''): h.update(block)
    return h.hexdigest()

def safe_download(url, path, max_mb=MAX_DOWNLOAD_MB):
    if path.exists() and path.stat().st_size>0:
        return {'status':'local_existing','bytes':path.stat().st_size, 'sha256':sha256file(path)}
    limit=int(max_mb*1024*1024); tmp=path.with_name(path.name+'.part')
    try:
        with requests.get(url,timeout=(15,90),stream=True,headers={'User-Agent':USER_AGENT}) as response:
            response.raise_for_status()
            length=int(response.headers.get('Content-Length','0') or 0)
            if length and length>limit: raise ValueError('File exceeds limit; download manually or use subset')
            received=0
            with open(tmp,'wb') as f:
                for block in response.iter_content(chunk_size=1024*256):
                    if not block: continue
                    received+=len(block)
                    if received>limit: raise ValueError('Download exceeds size cap')
                    f.write(block)
            if received==0: raise ValueError('Empty download')
            with open(tmp,'rb') as f: header=f.read(256).lower().lstrip()
            if header.startswith((b'<html',b'<!doctype html')):
                raise ValueError('Remote server supplied an HTML page, not a data file')
            if str(path).endswith('.gz'):
                with gzip.open(tmp,'rb') as f: f.read(10)
            tmp.replace(path)
            return {'status':'downloaded','bytes':received,'sha256':sha256file(path)}
    except Exception as exc:
        tmp.unlink(missing_ok=True)
        return {'status':'unavailable','error':str(exc)[:250]}

manifest=[]
for key,info in SOURCES.items():
    path=DATA/info['filename']
    result=safe_download(info['url'],path)
    row={'source':key,'url':info['url'],'path':str(path),**result}
    manifest.append(row)
    print(f"{key:30s} {result['status']:<16s} {result.get('bytes','')} {result.get('error','')[:55]}")
(OUT/'download_manifest.json').write_text(json.dumps(manifest,indent=2))
print('\nSource failures are recorded; they are NOT interpreted as zero biological overlap.')


### If a file is unavailable

1. Open the linked GEO record or [Dryad source](https://doi.org/10.5061/dryad.c59zw3rkj) manually and download the original file.
2. In Colab, open the Files sidebar and upload it into `/content/har_pilot/data/`, preserving the filename above.
3. Rerun from the next cell. **Do not rename or change the content to make a match appear.**

The GEO 305245 raw `*.rds` atlases are gigabyte-scale and intentionally **not** downloaded in this pilot. Sample metadata alone can be inspected initially.


In [ ]:
# 3. GEO sample audit: GSM sample counts are NOT necessarily biological replicate counts.
def parse_soft_samples(path):
    if not path.exists(): return pd.DataFrame()
    rows=[]; current=None
    try:
        with gzip.open(path,'rt',encoding='utf-8',errors='replace') as f:
            for line in f:
                line=line.strip()
                if line.startswith('^SAMPLE = '):
                    if current: rows.append(current)
                    current={'gsm':line.split('=',1)[1].strip(),'characteristics':[]}
                elif current is not None:
                    if line.startswith('!Sample_title = '): current['title']=line.split('=',1)[1].strip()
                    elif line.startswith('!Sample_source_name_ch1 = '): current['source_name']=line.split('=',1)[1].strip()
                    elif line.startswith('!Sample_characteristics_ch1 = '):
                        current['characteristics'].append(line.split('=',1)[1].strip())
        if current: rows.append(current)
    except Exception as e:
        print('Metadata parse issue:',path.name,str(e)[:100])
        return pd.DataFrame()
    for r in rows: r['characteristics']=' | '.join(r['characteristics'])
    return pd.DataFrame(rows)

frames=[]; meta_counts={}
for gse in ('GSE270828','GSE292632','GSE305245'):
    df=parse_soft_samples(DATA/f'{gse}_family.soft.gz')
    meta_counts[gse]=len(df) if len(df) else None
    if not df.empty:
        df.insert(0,'gse',gse); frames.append(df)
    print(gse,'GSM records:',len(df) if len(df) else 'unavailable')
    if not df.empty: print(df[['gsm','title','characteristics']].head(4).to_string(index=False,max_colwidth=95))
if frames:
    allmeta=pd.concat(frames,ignore_index=True)
    allmeta.to_csv(OUT/'GEO_sample_metadata.tsv',sep='\t',index=False)
else: allmeta=pd.DataFrame()

metadata_note={
 'GSM_sample_counts':meta_counts,
 'independent_donor_count':'NOT ESTABLISHED; must extract donor/line identifiers from study metadata',
 'warning':'Sequencing lanes, batches, guide-transduced cells, and GSM accessions are not automatically independent biological donors or lines.'
}
(OUT/'replication_audit.json').write_text(json.dumps(metadata_note,indent=2))


In [ ]:
# 4. Source file schemas and HAR identifier exploration

def read_delimited(path, is_bed=False):
    if not path.exists(): return None
    try:
        if is_bed:
            d=pd.read_csv(path,sep='\t',compression='infer',header=None,comment='#',dtype=str)
            if d.shape[1]<3: raise ValueError('Expected >=3 BED columns')
            return d
        d=pd.read_csv(path,sep=None,engine='python',dtype=str,on_bad_lines='skip')
        if d.shape[1]==1:
            alt=pd.read_csv(path,sep='\t',dtype=str,on_bad_lines='skip')
            if alt.shape[1]>1: d=alt
        return d
    except Exception as e:
        print('Cannot parse',path.name,str(e)[:160]); return None

features=read_delimited(DATA/'GSE270828_feature_README.csv')
target_bed=read_delimited(DATA/'GSE292632_NPC_4N_CRISPRi_screen_target_regions.bed.gz',is_bed=True)
contacts=read_delimited(DATA/'HAR_interactions_hNSCs.txt')
for label,df in [('GSE270828 feature README',features),('GSE292632 BED',target_bed),('Pal hNSC interactions',contacts)]:
    print('\n',label, 'unavailable' if df is None else f'{df.shape[0]:,} rows x {df.shape[1]} columns')
    if df is not None:
        print('columns:',list(map(str,df.columns))[:12]); print(df.head(2).to_string(index=False,max_colwidth=40))

def har_tags_from_df(df):
    if df is None: return set()
    # Keep naming systems distinct; e.g. HAR181 is not necessarily HARsv2_0181.
    patt=re.compile(r'(?<![A-Za-z0-9])(?:HARsv2[._-]?\d+|HACNS\d+|2xHAR[._-]?\d+|HAR[._-]?\d+)(?![A-Za-z0-9])',re.I)
    found=set()
    for c in df.columns:
        for value in df[c].dropna().astype(str):
            for match in patt.findall(value): found.add(match.upper())
    return found

feature_tags=har_tags_from_df(features)
contact_tags=har_tags_from_df(contacts)
name_overlap=sorted(feature_tags & contact_tags)
name_status=('not_established' if features is None or contacts is None else
             'no_exact_match_or_incompatible_naming' if not name_overlap else 'exact_label_overlap')
print('\nGSE270828 labels:',len(feature_tags),'Pal labels:',len(contact_tags))
print('EXACT shared labels:',len(name_overlap),'status:',name_status,'examples:',name_overlap[:10])
(OUT/'HAR_exact_name_overlap.json').write_text(json.dumps({
 'n_guide_HAR_strings':len(feature_tags),'n_chic_HAR_strings':len(contact_tags),
 'n_exact_label_matches':len(name_overlap),'name_overlap_status':name_status,
 'shared_labels':name_overlap,
 'warning':'Identical labels only; incompatible naming systems are not mapped without an authoritative crosswalk.'
},indent=2))


In [ ]:
# 5. Coordinate overlap between GSE292632-targeted regions and Pal hNSC HAR/HGE BAITS.
# This is *region* overlap, NOT validation of the same HAR perturbations or genes.
def chrom_norm(x):
    s=str(x).strip().lower()
    return s[3:] if s.startswith('chr') else s

def make_pal_baits(df):
    needed=['Bait_chr','Bait_start','Bait_end','Bait_name']
    if df is None: return None,'Pal table unavailable'
    columns={str(c).strip().lower():c for c in df.columns}
    if not all(c.lower() in columns for c in needed): return None, 'Pal headers missing: '+str(needed)
    d=df[[columns[x.lower()] for x in needed]].copy(); d.columns=['chrom','start','end','har_label']
    d['start']=pd.to_numeric(d['start'],errors='coerce')
    d['end']=pd.to_numeric(d['end'],errors='coerce')
    d=d.dropna(subset=['chrom','start','end'])
    d=d[d.end>d.start].copy(); d[['start','end']]=d[['start','end']].astype(int)
    d['chrom']=d.chrom.map(chrom_norm)
    return d.drop_duplicates(),None

def make_bed_targets(df):
    if df is None: return None,'GSE292632 BED unavailable'
    d=df.iloc[:,:min(4,df.shape[1])].copy()
    d.columns=['chrom','start','end']+(['label'] if d.shape[1]>3 else [])
    if 'label' not in d: d['label']='region_'+(d.index+1).astype(str)
    d['start']=pd.to_numeric(d['start'],errors='coerce')
    d['end']=pd.to_numeric(d['end'],errors='coerce')
    d=d.dropna(subset=['start','end','chrom']); d=d[d.end>d.start].copy()
    d[['start','end']]=d[['start','end']].astype(int); d['chrom']=d.chrom.map(chrom_norm)
    return d.drop_duplicates(),None

pal_baits,err1=make_pal_baits(contacts); screen_regions,err2=make_bed_targets(target_bed)
coord_report={
 'Pal_hNSC_bait_rows':None if pal_baits is None else len(pal_baits),
 'GSE292632_target_rows':None if screen_regions is None else len(screen_regions),
 'Pal_build':VERIFIED_PAL_HNSC_BUILD,'GSE292632_bed_build':VERIFIED_GSE292632_BED_BUILD,
 'status':'NOT_RUN','note':''
}
matched=[]
if err1 or err2:
    coord_report.update(status='source_missing_or_bad_schema',note='; '.join(x for x in [err1,err2] if x))
elif VERIFIED_GSE292632_BED_BUILD is None:
    coord_report.update(status='BLOCKED_UNVERIFIED_GENOME_BUILD',
      note='Verify the 106-region BED coordinate reference from original metadata before setting VERIFIED_GSE292632_BED_BUILD = GRCh38')
elif VERIFIED_GSE292632_BED_BUILD != VERIFIED_PAL_HNSC_BUILD:
    coord_report.update(status='BLOCKED_GENOME_BUILD_MISMATCH',note='Liftover must be independently documented; do not compare unmatched coordinate assemblies')
else:
    # 106 targeted regions x unique HAR/HGE bait fragments (fast enough without genomic libraries)
    by_chr={ch:g for ch,g in pal_baits.groupby('chrom')}
    for row in screen_regions.itertuples(index=False):
        subset=by_chr.get(row.chrom)
        if subset is None: continue
        hits=subset[(subset.start<row.end)&(subset.end>row.start)]
        for hit in hits.itertuples(index=False):
            matched.append({'target_region':row.label,'chrom':row.chrom,'target_start':row.start,
                            'target_end':row.end,'Pal_bait_label':hit.har_label,
                            'bait_start':hit.start,'bait_end':hit.end})
    coord_report.update(status='RUN_VALID_SAME_BUILD',number_of_overlap_pairs=len(matched),
        number_of_overlapping_GSE292632_regions=len(set(r['target_region'] for r in matched)),
        number_of_overlap_bait_names=len(set(r['Pal_bait_label'] for r in matched)),
        note='Physical intersection only; different perturbation sets and functional effects NOT established')
    pd.DataFrame(matched,columns=['target_region','chrom','target_start','target_end',
                      'Pal_bait_label','bait_start','bait_end']).to_csv(OUT/'region_overlap_GSE292632_vs_Pal.tsv',sep='\t',index=False)
(OUT/'coordinate_overlap_audit.json').write_text(json.dumps(coord_report,indent=2))
print(json.dumps(coord_report,indent=2))


In [ ]:
# 6. Produce a conservative feasibility report + downloadable ZIP
status=['# Feasibility pilot report','',f"Run date (UTC): {datetime.now(timezone.utc).isoformat()}",
'','## Data acquisition']
for r in manifest: status.append(f"- {r['source']}: {r['status']}"+(f" ({r.get('error')})" if r.get('error') else ''))
status+=['','## Biological replication']
for gse,n in meta_counts.items(): status.append(f'- {gse}: '+('GSM metadata not fetched' if n is None else f'{n} GSM sample records (NOT independent donors)'))
status+=['- Confirmed independent donor/line number: NOT ESTABLISHED from this pilot.',
'', '## Cross-dataset evidence',
f'- HAR exact label-match status: {name_status}. Shared exact HAR strings: {len(name_overlap)} (not validated crosswalk).',
f"- Genomic overlap status: {coord_report['status']}",
'- No differential perturbation effect sizes were computed from guide-feature metadata.',
'- No claim of experimental replication, target-gene causality or developmental phenotype is made.',
'', '## Next required steps',
'1. Verify genomic build and HAR name crosswalk using original supplementary guide design and reference FASTA/BED.',
'2. Obtain a tested HAR–gene perturbation-effect table, including negative controls and guide-to-HAR mapping.',
'3. Inspect independent donor/line and batch identifiers; specify valid experimental unit.',
'4. Only then perform held-out HAR-locus benchmark and developmental atlas context analysis.',
'5. Retain all NA/unavailable states; never report missing files as zero overlap.']
(OUT/'PILOT_FEASIBILITY_REPORT.md').write_text('\n'.join(status)+'\n',encoding='utf-8')
zip_path=BASE/'HAR_Pilot_Run_Outputs.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.glob('*'): z.write(p,p.name)
print('\n'.join(status))
print('\nAudit output:',zip_path)
try:
    from google.colab import files
    print('In Colab, run files.download(str(zip_path)) to download the audit ZIP.')
except ImportError: pass
